# WWR Segmentation — Google Colab (A100)

### Before running
1. **Runtime → Change runtime type → A100 GPU**
2. Upload `data.zip` to: `My Drive/WWR_Seg_Model/data.zip`
3. **Run cells in order** — cell 1 loads the code automatically

> Cell 1 clones from GitHub if the project is not on Drive.  
> Push latest code to GitHub first, **or** upload the repo to `My Drive/WWR_Seg_Model/`.

In [ ]:
# Install dependencies
!pip install -q tensorflow>=2.15 numpy pandas matplotlib scikit-learn openpyxl

## 1. Mount Drive & Load Project Code (required — run before any import)

In [ ]:
import os
import sys
import subprocess
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/kermanimohammad/U-Net_Segmentation.git"
LOCAL_REPO = Path("/content/U-Net_Segmentation")

# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# ── Find WWR_Segmentation on Drive or clone from GitHub ─────────────────────
def _find_project_root() -> Path | None:
    for candidate in (
        LOCAL_REPO,
        Path("/content/drive/MyDrive/WWR_Seg_Model/code"),
        Path("/content/drive/MyDrive/WWR_Seg_Model/U-Net_Segmentation"),
        Path("/content/drive/MyDrive/WWR_Seg_Model"),
    ):
        if (candidate / "WWR_Segmentation" / "__init__.py").exists():
            return candidate.resolve()
    return None

project_root = _find_project_root()

if project_root is None:
    print(f"Project not found on Drive — cloning from GitHub...")
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(LOCAL_REPO)],
        check=True,
    )
    project_root = LOCAL_REPO

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root : {project_root}")
print(f"Package found: {(project_root / 'WWR_Segmentation').exists()}")

## 2. Colab Setup — Mount Drive & Unzip data.zip to Local Storage

In [ ]:
from WWR_Segmentation.colab_setup import setup_colab
from WWR_Segmentation.dataset import get_dataset_info

# clone_if_missing=False because cell 1 already loaded the project
config = setup_colab(force_unzip=False, clone_if_missing=False)

get_dataset_info(config)

## 3. Training

In [ ]:
from WWR_Segmentation.trainer import train

model, history = train(config)

## 4. Evaluation (Validation + Independent Test Set)

In [ ]:
from WWR_Segmentation.evaluate import run_evaluation

eval_results = run_evaluation(config)
print(f"Test mean IoU: {eval_results['test']['metrics']['mean_iou']['value']:.4f}")

## 5. Window-to-Wall Ratio (WWR)

In [ ]:
from WWR_Segmentation.wwr import run_wwr_analysis

wwr_df = run_wwr_analysis(config)
wwr_df.head()

## 6. Optional: 5-Fold Cross-Validation

In [ ]:
# Uncomment to run (train set only — test set is never touched)
# from WWR_Segmentation.cross_validation import run_cross_validation
# cv_summary = run_cross_validation(config)

## 7. Inference on Test Images

In [ ]:
from WWR_Segmentation.inference import run_inference

run_inference(config, input_dir=config.test_images_dir)